# Global Scenario Forecaster Training

This notebook trains a global Transformer forecaster on the TRAIN_SKUS split and saves it for reuse in `dif-pi`.

- Model: `ScenarioGenerationTransformerForecaster`
- Training: pooled sliding windows across TRAIN_SKUS (univariate demand)
- Inference in DIF‑PI: per‑SKU scaler fitted on the selected SKU's history (no retraining)


## 1) Environment and training configuration

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys
from src.transformer_forecaster import ScenarioGenerationTransformerForecaster
from sklearn.model_selection import train_test_split

In [ ]:
REPO_ROOT = Path('.').resolve()

# Panel exported by the EDA notebook
PANEL_PATH = REPO_ROOT / 'datasets' / 'processed' / 'difpi_pricing_demand_panel.csv'

# Where to save the global model (directory)
MODEL_DIR = REPO_ROOT / 'artifacts' / 'models' / 'scenario_gen_transformer_global'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Columns
SKU_COL='StockCode'
TIME_COL='timestamp'
DEMAND_COL='demand'

# Eligibility filters
ELIGIBILITY_MODE = "adaptive"  # {"strict","relaxed","adaptive"}
MIN_HISTORY_DAYS_STRICT = 365
MIN_NONZERO_DAYS_STRICT = 60
MIN_HISTORY_DAYS_RELAXED = 120
MIN_NONZERO_DAYS_RELAXED = 30
TARGET_ELIGIBLE_FRACTION = 0.8

# SKU holdout split (train/test)
SKU_HOLDOUT_ENABLED = True
TEST_SKU_FRAC = 0.20
SPLIT_SEED = 42

# Global forecaster hyperparameters
SEQ_LEN = 30
# Global forecaster data requirement
# IMPORTANT: zeros are allowed; min_points refers to total rows, not non-zero days.
MIN_POINTS_PER_SKU = 2*SEQ_LEN + 2  # pooled training needs enough windows per SKU

BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-3

# Optional: use only the first X% of each TRAIN_SKU (drop tail)
PER_SKU_TRAIN_FRAC = 1.0   # set 0.9 if you want to drop the last 10% of each train SKU

SEED = 42
np.random.seed(SEED)

print("PANEL_PATH:", PANEL_PATH)
print("MODEL_DIR:", MODEL_DIR)

PANEL_PATH: /Users/alexgrigoras/Library/Mobile Documents/com~apple~CloudDocs/[5] Software/github/dif-pi/datasets/processed/difpi_pricing_demand_panel.csv
MODEL_DIR: /Users/alexgrigoras/Library/Mobile Documents/com~apple~CloudDocs/[5] Software/github/dif-pi/artifacts/models/scenario_gen_transformer_global


## 2) Load panel + build eligible SKU pool

In [ ]:
if not PANEL_PATH.exists():
    raise FileNotFoundError(f"Missing {PANEL_PATH}. Run the EDA export notebook first.")

panel = pd.read_csv(PANEL_PATH)
panel[SKU_COL] = panel[SKU_COL].astype(str)
panel[TIME_COL] = pd.to_datetime(panel[TIME_COL], errors='raise')

def sku_stats(df):
    g = df.groupby(SKU_COL)
    out = g.agg(
        n_rows=(DEMAND_COL, 'size'),
        n_nonzero=(DEMAND_COL, lambda x: int((np.asarray(x, float) > 0).sum())),
        min_time=(TIME_COL, 'min'),
        max_time=(TIME_COL, 'max')
    ).reset_index()
    out['history_days'] = (out['max_time'] - out['min_time']).dt.days + 1
    return out

stats = sku_stats(panel)

def apply_thresholds(stats_df, min_hist_days, min_nonzero_days):
    mask = (stats_df['history_days'] >= min_hist_days) & (stats_df['n_nonzero'] >= min_nonzero_days)
    return stats_df.loc[mask].copy()

def adaptive_thresholds(stats_df):
    # stepwise relaxation to reach target fraction
    steps = [
        (MIN_HISTORY_DAYS_STRICT, MIN_NONZERO_DAYS_STRICT),
        (270, 45),
        (180, 30),
        (MIN_HISTORY_DAYS_RELAXED, MIN_NONZERO_DAYS_RELAXED),
        (90, 20),
        (60, 15),
    ]
    total = len(stats_df)
    for mh, mnz in steps:
        elig = apply_thresholds(stats_df, mh, mnz)
        frac = len(elig) / total if total else 0
        if (ELIGIBILITY_MODE != "adaptive") or (frac >= TARGET_ELIGIBLE_FRACTION) or ((mh, mnz) == steps[-1]):
            return elig, mh, mnz, frac
    return elig, mh, mnz, frac

if ELIGIBILITY_MODE == "strict":
    eligible = apply_thresholds(stats, MIN_HISTORY_DAYS_STRICT, MIN_NONZERO_DAYS_STRICT)
    used_hist, used_nnz = MIN_HISTORY_DAYS_STRICT, MIN_NONZERO_DAYS_STRICT
    used_frac = len(eligible) / len(stats) if len(stats) else 0
elif ELIGIBILITY_MODE == "relaxed":
    eligible = apply_thresholds(stats, MIN_HISTORY_DAYS_RELAXED, MIN_NONZERO_DAYS_RELAXED)
    used_hist, used_nnz = MIN_HISTORY_DAYS_RELAXED, MIN_NONZERO_DAYS_RELAXED
    used_frac = len(eligible) / len(stats) if len(stats) else 0
else:
    eligible, used_hist, used_nnz, used_frac = adaptive_thresholds(stats)

print(f"Eligible SKUs: {len(eligible)} / {len(stats)} (mode={ELIGIBILITY_MODE}, min_history_days={used_hist}, min_nonzero_days={used_nnz}, frac={used_frac:.2f})")
eligible_skus = eligible[SKU_COL].astype(str).tolist()

Eligible SKUs: 100 / 100 (mode=adaptive, min_history_days=365, min_nonzero_days=60, frac=1.00)


## 3) Split SKUs into TRAIN/TEST and train global forecaster

In [ ]:
if not SKU_HOLDOUT_ENABLED:
    raise ValueError("SKU_HOLDOUT_ENABLED must be True for this global-forecaster training notebook.")

train_skus, test_skus = train_test_split(
    eligible_skus,
    test_size=TEST_SKU_FRAC,
    random_state=SPLIT_SEED,
    shuffle=True
)

print("TRAIN_SKUS:", len(train_skus))
print("TEST_SKUS:", len(test_skus))

# Save the splits for reuse
MODEL_DIR.mkdir(parents=True, exist_ok=True)
pd.Series(train_skus, name="sku").to_csv(MODEL_DIR / "train_skus.csv", index=False)
pd.Series(test_skus, name="sku").to_csv(MODEL_DIR / "test_skus.csv", index=False)
print("Saved train/test SKU lists to:", MODEL_DIR)

TRAIN_SKUS: 80
TEST_SKUS: 20
Saved train/test SKU lists to: /Users/alexgrigoras/Library/Mobile Documents/com~apple~CloudDocs/[5] Software/github/dif-pi/artifacts/models/scenario_gen_transformer_global


In [ ]:
# Diagnostics: per-SKU history length in TRAIN_SKUS
len_df = (
    panel.loc[panel[SKU_COL].isin(train_skus)]
         .groupby(SKU_COL)[DEMAND_COL]
         .size()
         .reset_index(name="n_rows")
         .sort_values("n_rows", ascending=False)
)
print("TRAIN_SKUS rows summary:")
print(len_df["n_rows"].describe())

print("\nTop 10 TRAIN_SKUS by rows:")
print(len_df.head(10).to_string(index=False))

print("\nBottom 10 TRAIN_SKUS by rows:")
print(len_df.tail(10).to_string(index=False))

TRAIN_SKUS rows summary:
count     80.000000
mean     706.000000
std        9.872606
min      632.000000
25%      705.000000
50%      709.000000
75%      710.000000
max      711.000000
Name: n_rows, dtype: float64

Top 10 TRAIN_SKUS by rows:
StockCode  n_rows
  1004906     711
  1082185     711
   995785     711
   981760     711
   930917     711
   866211     711
   860776     711
   854852     711
   833715     711
   826249     711

Bottom 10 TRAIN_SKUS by rows:
StockCode  n_rows
  1120741     702
  1007195     702
  1022003     701
   903325     701
  6534178     700
   911878     693
  6534166     690
   962229     686
   397896     684
  6544236     632


In [ ]:
sys.path.append(str(REPO_ROOT))

fg = ScenarioGenerationTransformerForecaster(sequence_length=SEQ_LEN)

info = fg.fit_global(
    panel_df=panel,
    sku_col=SKU_COL,
    time_col=TIME_COL,
    target_col=DEMAND_COL,
    train_skus=train_skus,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=LR,
    per_sku_train_frac=PER_SKU_TRAIN_FRAC,
    min_points=MIN_POINTS_PER_SKU,
    verbose=1
)

print("Training info:", {k: info[k] for k in ["n_train_skus","n_used_skus","n_skipped_skus","n_train_windows","sequence_length"]})

Epoch 1/50
845/845 ━━━━━━━━━━━━━━━━━━━━ 65s 75ms/step - loss: 0.0394 - learning_rate: 0.0010
Epoch 2/50
845/845 ━━━━━━━━━━━━━━━━━━━━ 64s 75ms/step - loss: 0.0095 - learning_rate: 0.0010
Epoch 3/50
845/845 ━━━━━━━━━━━━━━━━━━━━ 61s 72ms/step - loss: 0.0092 - learning_rate: 0.0010
Epoch 4/50
845/845 ━━━━━━━━━━━━━━━━━━━━ 60s 72ms/step - loss: 0.0092 - learning_rate: 0.0010
Epoch 5/50
845/845 ━━━━━━━━━━━━━━━━━━━━ 64s 76ms/step - loss: 0.0090 - learning_rate: 0.0010
Epoch 6/50
845/845 ━━━━━━━━━━━━━━━━━━━━ 62s 73ms/step - loss: 0.0089 - learning_rate: 0.0010
Epoch 7/50
845/845 ━━━━━━━━━━━━━━━━━━━━ 62s 73ms/step - loss: 0.0089 - learning_rate: 0.0010
Epoch 8/50
845/845 ━━━━━━━━━━━━━━━━━━━━ 62s 73ms/step - loss: 0.0089 - learning_rate: 0.0010
Epoch 9/50
845/845 ━━━━━━━━━━━━━━━━━━━━ 61s 73ms/step - loss: 0.0089 - learning_rate: 0.0010
Epoch 10/50
845/845 ━━━━━━━━━━━━━━━━━━━━ 90s 106ms/step - loss: 0.0089 - learning_rate: 0.0010
Epoch 11/50
845/845 ━━━━━━━━━━━━━━━━━━━━ 311s 368ms/step - loss: 0.0

In [ ]:
# Saving the trained model
saved_path = fg.save(MODEL_DIR)
print("Saved global forecaster to:", saved_path)

Saved global forecaster to: /Users/alexgrigoras/Library/Mobile Documents/com~apple~CloudDocs/[5] Software/github/dif-pi/artifacts/models/scenario_gen_transformer_global/scenario_gen_transformer.keras


## 4) Quick forecasting test

In [ ]:
fg2 = ScenarioGenerationTransformerForecaster.load(MODEL_DIR)

# choose one test SKU and forecast 30 steps using its history (no retraining)
case_sku = test_skus[0]
s = panel.loc[panel[SKU_COL].astype(str) == str(case_sku)].sort_values(TIME_COL)
y_hist = s[DEMAND_COL].astype(float).values
pred = fg2.forecast(y_hist, forecast_length=30)

print("CASE_SKU:", case_sku, "history_len:", len(y_hist), "pred_len:", len(pred))
print("pred[:5] =", pred[:5])

CASE_SKU: 951590 history_len: 711 pred_len: 30
pred[:5] = [11.743623   9.726634   8.3955145  8.836785   8.443537 ]
